# Prepare proteomic phenotypes and covariates for pQTL analysis

This notebook prepares All of Us (AoU) proteomics phenotypes and covariates for pQTL analysis. It assumes AoU genetic ancestry assignments and the CDR version 9 (CDRv9) proteomics release. The workflow filters samples with complete assay measurements, identifies proteomic outliers, maps protein identifiers to GENCODE gene coordinates, rank-normalizes phenotypes within ancestry groups, computes proteomic principal components, and combines them with genetic principal components.

Expected inputs and outputs should be versioned alongside the analysis. Population labels follow the AoU genetic ancestry assignments: `AFR`, `AMR`, `EAS`, `EUR`, `MID`, `SAS`, and `COMB` for the combined analysis. Update the input paths, release identifiers, and output locations before running this notebook.

Originally developed by Evin Padhi (Stanford University).

In [ ]:
install.packages("Hmisc", repos = "https://cloud.r-project.org")

In [ ]:
# --- install/load packages robustly ---
cran_pkgs <- c("tidyverse", "data.table", "patchwork", "magrittr", "WGCNA", "biomaRt")

# OlinkAnalyze can be CRAN/Bioc depending on version/environment; we’ll try CRAN then fallback.
extra_pkgs <- c("OlinkAnalyze")

install_if_missing <- function(pkgs) {
  missing <- pkgs[!vapply(pkgs, requireNamespace, FUN.VALUE = logical(1), quietly = TRUE)]
  if (length(missing)) install.packages(missing, repos = "https://cloud.r-project.org")
}

install_if_missing(cran_pkgs)
install_if_missing(extra_pkgs)

BiocManager::install(c("impute", "preprocessCore", "GO.db", "AnnotationDbi", "Hmisc", "WGCNA"),
                      update = FALSE, ask = FALSE)

# biomaRt is typically Bioconductor; ensure it's available even if CRAN install didn't work.
if (!requireNamespace("biomaRt", quietly = TRUE)) {
  if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager", repos = "https://cloud.r-project.org")
  BiocManager::install("biomaRt", update = FALSE, ask = FALSE)
}

# OlinkAnalyze fallback if not installed
if (!requireNamespace("OlinkAnalyze", quietly = TRUE)) {
  # Try Bioconductor first (some environments)
  if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager", repos = "https://cloud.r-project.org")
  try(BiocManager::install("OlinkAnalyze", update = FALSE, ask = FALSE), silent = TRUE)
}


In [ ]:
# Moving combined psam into VM
system("gsutil -m cp gs://path/to/COMB.psam ./")


# Move normalized reps removed matrix into notebook
system("gsutil -m cp gs://path/to/reps_removed.tsv.gz ./")

In [ ]:
library(tidyverse)
library(data.table)
library(data.table)
library(patchwork)
library(OlinkAnalyze)
library(magrittr)
library(WGCNA)
library(biomaRt)

theme_set(theme_classic())
theme_update(panel.border = element_rect(fill = NA,linewidth = .9), 
            axis.line = element_blank(),
            axis.text = element_text(size = 15),
            axis.title = element_text(size = 20),
            strip.background = element_blank(),
            strip.text = element_text(size = 12))

multiomics_sample_list <- fread('COMB.psam') %>% dplyr::rename('ResearchID' = 1)
olink_median_normalized_path <- 'reps_removed.tsv.gz'
olink_df <- fread(olink_median_normalized_path) %>% filter(ResearchID %in% multiomics_sample_list$ResearchID)

## Map UniProt protein identifiers to Ensembl gene identifiers

This mapping is used to attach genomic coordinates to the proteomic features. The Ensembl release and mapping date should be recorded because identifier mappings can change over time.

In [ ]:
library(biomaRt)

protein_list <- olink_df %>% dplyr::select(UniProt) %>% distinct() %>% pull(UniProt)

mart <- useEnsembl(
  biomart = "genes",
  dataset = "hsapiens_gene_ensembl"
)

conversion_list <- getBM(
  attributes = c("ensembl_gene_id_version", "uniprot_gn_id"),
  filters    = "uniprot_gn_id",
  values     = protein_list,
  mart       = mart
) %>%
  dplyr::rename(gene_id = 1, UniProt = 2)

## Compute assay and sample missingness

Use `PCNormalizedNPX` to assess missingness because it represents the original normalized assay values before the downstream outlier-analysis step. The later outlier analysis uses the batch-corrected `NPX` values. This distinction should remain explicit because the two columns represent different processing stages.

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 12, repr.plot.res = 200)


# get number of assays that have a missing value per protein 
missingness_per_assay <- olink_df%>% 
        group_by(Assay) %>% 
        summarize(n = 100*sum(is.na(PCNormalizedNPX))/dplyr::n()) 

missingness_per_assay_plot <- missingness_per_assay %>% 
        ggplot(aes(x = n)) + 
            geom_histogram() + 
            xlab('% Missing People') +
            ylab('Number Assays') 

# compute missingness per person
missingness_per_person <- olink_df %>% 
        group_by(SampleID) %>% 
        summarize(n = sum(is.na(PCNormalizedNPX))) 


missingness_per_person_plot <- missingness_per_person %>%  
        ggplot(aes(x = n)) + 
            geom_histogram() + 
            xlab('Number Missing Assays') +
            ylab('Number People') 
missingness_per_assay_plot + missingness_per_person_plot


person_keep_list <- missingness_per_person %>% filter(n < 1)
print(paste0('Number of individuals with no missing assays ', person_keep_list %>% nrow))

The retained sample count is reported by the preceding code cell so it stays synchronized with the selected CDRv9 input release.

## Convert the proteomics matrix to wide format and identify sample outliers with WGCNA

In [ ]:
# Let's look at the normalized values and see which is which
colnames(olink_df)

In [ ]:
NPX_data_wide <- olink_df %>% 
    filter(SampleID %in% person_keep_list$SampleID) %>% 
    group_by(ResearchID,OlinkID) %>% 
    filter(row_number() == 1) %>% 
    ungroup() %>% 
    filter(Assay != 'GBP1' & Assay != 'MAP2K1') %>% 
    dplyr::select(ResearchID,NPX,UniProt) %>% 
    pivot_wider(names_from = UniProt,values_from =NPX )  %>% 
    column_to_rownames('ResearchID')


In [ ]:
# compute connectivity score by calculating correlation 
# of all samples with each other and then Z-scoring the data
norm_adj <- (0.5 + 0.5 * bicor(NPX_data_wide %>% t()))
net_summary <- fundamentalNetworkConcepts(norm_adj)
net_connectivity <- net_summary$Connectivity
connectivity_zscore <- ((net_connectivity - mean(net_connectivity)) / sd(net_connectivity)) %>% 
        data.frame()  %>% 
        dplyr::rename('Z_score' = 1) %>% 
        rownames_to_column('ResearchID')

connectivity_zscore  %>% head
connectivity_zscore_outliers <- connectivity_zscore %>% filter(Z_score < -3) 

print(paste0('Number of Connectivity Z score outliers ',connectivity_zscore_outliers %>% nrow))

# # commented following out just so we don't overwirte
connectivity_zscore %>% write_tsv('connectivity_z_scores.tsv')

connectivity_zscore_outliers %>% dplyr::select(ResearchID) %>% write_tsv('connectivity_z_score_outliers.tsv', col_names=FALSE)

system(paste0('gsutil cp connectivity_z_score_outliers.tsv gs://path/to/output_folder/'))
system(paste0('gsutil cp connectivity_z_scores.tsv gs://path/to/output_folder/'))

The number of flagged outliers is reported by the preceding code cell and may vary with the input release and identifier mapping.

In [ ]:
connectivity_zscore_outliers <- fread(
  'connectivity_z_score_outliers.tsv',
  header = FALSE,
  col.names = "ResearchID"
)

PCNPX_wide_filtered <- NPX_data_wide %>% 
    rownames_to_column('ResearchID') %>% 
    filter(!ResearchID %in% connectivity_zscore_outliers$ResearchID) %>% 
    column_to_rownames('ResearchID') 

print(paste0('Number of samples after removing outliers:',PCNPX_wide_filtered %>% nrow))
rownames(PCNPX_wide_filtered) %>% data.frame() %>% write_tsv('sample_list.tsv',col_names = FALSE)

The retained sample count is reported by the preceding code cell and may vary with the CDRv9 input file and outlier list.

## Map protein identifiers to gene coordinates

The workflow uses Ensembl release 114 and a GENCODE v48 GRCh38 gene annotation. Record these annotation versions with any released output.

In [ ]:
library(biomaRt)

ensembl <- useEnsembl(
  biomart = "genes",
  dataset = "hsapiens_gene_ensembl",
  version = 114
)

In [ ]:
protein_list <- olink_df %>%
  dplyr::select(UniProt) %>%
  distinct() %>%
  pull(UniProt)

conversion_list <- biomaRt::getBM(
  attributes = c("ensembl_gene_id_version", "uniprot_gn_id"),
  filters    = "uniprot_gn_id",
  values     = protein_list,
  mart       = ensembl
) %>%
  dplyr::rename(
    gene_id = ensembl_gene_id_version,
    UniProt = uniprot_gn_id
  )

In [ ]:
# extract transcription start sites from gencode GTF file, bring this in from RNA seq workspace 
gencode_gtf_path <-'gs://path/to/gene_annotation/gencode.v48.GRCh38.genes.collapsed_only.gtf'
system(paste0('gsutil cp ',gencode_gtf_path, ' .'))
gencode_GTF <- rtracklayer::import('gencode.v48.GRCh38.genes.collapsed_only.gtf') %>% data.frame()

# map TSS locations based on strand
TSS_locations <- gencode_GTF  %>% 
    filter(type == 'gene'  ) %>%
    mutate(TSS = case_when(strand == '+' ~ start,TRUE ~ end)) %>% 
    dplyr::select(gene_id,TSS,seqnames) %>% 
    mutate(start = TSS -1,end = TSS) %>% 
    dplyr::select(gene_id,start,end,seqnames)


# joins normalized NPX data with ensembl gene id uniprot conversion table 
# and then joins with transcription start site data from the GTF file. 
# Note: Some Uniprot IDs map back to multiple genes. Im just leaving 
# these in for now since these are likely complexes

gene_id_merged_NPX_data <- PCNPX_wide_filtered %>%
    t() %>% 
    data.frame() %>% 
    rownames_to_column('UniProt') %>% 
    left_join(conversion_list,by = 'UniProt') %>% 
    left_join(TSS_locations,by = 'gene_id') %>%
    dplyr::select(seqnames,start,end,UniProt,gene_id,everything()) %>% 
    filter(!is.na(seqnames)) %>% 
    dplyr::rename_with(~str_remove(.,'X')) %>% 
    mutate(gene_id = paste0(UniProt,'_',gene_id)) %>% 
    dplyr::select(-UniProt) %>% 
    arrange(seqnames,start)

gene_id_merged_NPX_data %>% head

## Rank-normalize phenotypes within ancestry groups and compute covariates

In [ ]:
# Bring in the ancestry predictions
ancestry_prediction_path <- 'gs://path/to/ancestry_preds.tsv'
system(paste0('gsutil cp ',ancestry_prediction_path, ' .'))

ancestry_df <- fread('ancestry_preds.tsv') %>% dplyr::select(research_id,ancestry_pred_other)

In [ ]:
write_covars_to_bucket <- function(ancestry_df,
                                   group_label,
                                   bed_data,
                                   bed_bucket_path='',
                                   covar_bucket_path = '',
                                   covariate_suffix ='',
                                   bed_suffix='',
                                   filter= TRUE
                                      ){

# set up output file names 
print(paste0('Running on population ',group_label))
bed_file_name <- paste0('AoU_',group_label,'_',bed_suffix)
covar_file_name <- paste0('AoU_',group_label,'_',covariate_suffix)

# print output file names 
print(paste0('Output bed file name:',bed_file_name))
print(paste0('Output covariate file name',covar_file_name))

if (filter == TRUE){
# get ancestries research IDs from file 
group_ids <- ancestry_df %>% filter(ancestry_pred_other == group_label) %>% mutate(research_id = as.character(research_id))%>% pull(research_id)
} else {
group_ids <- ancestry_df %>% mutate(research_id = as.character(research_id))%>% pull(research_id)
    
}
    
# subsets phenotype bed to population of interest and 
# further breaks that down into metadata and quantification 
# values. Then use INT to nornmalize quantifications within 
# the group and merges data back together for PCA analysis 
# and downstream QTL calling 
subset_bed <- bed_data %>% dplyr::select(1:4,any_of(group_ids))
meta_data <- subset_bed %>%  dplyr::select(1:4)

normalized_quantifications <- subset_bed %>% 
                dplyr::select(any_of(group_ids)) %>% 
                t() %>% 
                data.frame() %>%  
                mutate(across(everything(),~RNOmni::RankNorm(.))) %>% 
                t() %>% 
                data.frame() %>% 
                arrange()
rownames(normalized_quantifications) <- NULL
merged_data <- bind_cols(meta_data,normalized_quantifications) %>% 
    dplyr::rename_with(~str_remove(.,'X')) %>% 
    dplyr::rename('#chr'='seqnames')
merged_data %>% fwrite(bed_file_name,sep='\t')


PCA_res <- merged_data %>% 
            data.frame() %>% 
            column_to_rownames('gene_id') %>% 
            dplyr::select(-1,-2,-3) %>% 
            PCAtools::pca()
num_PCs <- PCAtools::chooseGavishDonoho( normalized_quantifications  ,  var.explained = PCA_res$sdev^2, noise = 1)


print(paste0('number of PCs chosen ',num_PCs))
PCA_calculations <- PCA_res$rotated %>%
        dplyr::select(1:num_PCs) %>% 
        data.frame() %>% 
        rownames_to_column('research_id') %>%  
        mutate(research_id = str_remove(research_id,'X'))

PCA_calculations %>% fwrite(covar_file_name,sep='\t')

    
print('Copying covariate file to bucket')
system(paste0('gsutil cp ' , covar_file_name, ' ',covar_bucket_path))
    
    
print('Copying subset bed file to bucket')    
system(paste0('gsutil cp ' , bed_file_name, ' ',bed_bucket_path))   
    
 
# gets the number of samples in the bed and covariate data
number_samples_bed <- subset_bed %>% dplyr::select(-1,-2,-3,-4) %>% ncol
  
print('Number samples in subset bed')
print(number_samples_bed)
}

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install("PCAtools")

In [ ]:
# install.packages("RNOmni")
library(RNOmni)

In [ ]:
bed_bucket_path <- 'gs://path/to/output/bed_files/'
covar_bucket_path <- 'gs://path/to/output/phenotype_covariates/phenotype_PCs/'
covariate_suffix <- 'pQTL_phenotype_PCs.tsv'
bed_suffix <- 'pQTL.bed.gz'
write_covars_to_bucket(ancestry_df,
                       'mid',
                       gene_id_merged_NPX_data,
                       bed_bucket_path = bed_bucket_path,
                       covar_bucket_path = covar_bucket_path,
                       covariate_suffix = covariate_suffix,
                       bed_suffix = bed_suffix)
write_covars_to_bucket(ancestry_df,
                       'afr',
                       gene_id_merged_NPX_data,
                       bed_bucket_path = bed_bucket_path,
                       covar_bucket_path = covar_bucket_path,
                       covariate_suffix = covariate_suffix,
                       bed_suffix = bed_suffix)
write_covars_to_bucket(ancestry_df,
                       'eur',
                       gene_id_merged_NPX_data,
                       bed_bucket_path = bed_bucket_path,
                       covar_bucket_path = covar_bucket_path,
                       covariate_suffix = covariate_suffix,
                       bed_suffix = bed_suffix)
write_covars_to_bucket(ancestry_df,
                       'sas',
                       gene_id_merged_NPX_data,
                       bed_bucket_path = bed_bucket_path,
                       covar_bucket_path = covar_bucket_path,
                       covariate_suffix = covariate_suffix,
                       bed_suffix = bed_suffix)
write_covars_to_bucket(ancestry_df,
                       'eas',
                       gene_id_merged_NPX_data,
                       bed_bucket_path = bed_bucket_path,
                       covar_bucket_path = covar_bucket_path,
                       covariate_suffix = covariate_suffix,
                       bed_suffix = bed_suffix)
write_covars_to_bucket(ancestry_df,
                       'amr',
                       gene_id_merged_NPX_data %>% distinct(),
                       bed_bucket_path = bed_bucket_path,
                       covar_bucket_path = covar_bucket_path,
                       covariate_suffix = covariate_suffix,
                       bed_suffix = bed_suffix)
write_covars_to_bucket(ancestry_df,
                       'comb',
                       gene_id_merged_NPX_data,
                       bed_bucket_path = bed_bucket_path,
                       covar_bucket_path = covar_bucket_path,
                       covariate_suffix = covariate_suffix,
                       bed_suffix = bed_suffix,
                       filter = FALSE)

## Merge proteomic and genetic principal components

Join covariates by participant identifier within each AoU genetic ancestry group. The combined output is intended for downstream pQTL association analyses.

In [ ]:
mid_genetic_PCs_path <- 'gs://path/to/MID_genetic_PCs.tsv'
afr_genetic_PCs_path <- 'gs://path/to/AFR_genetic_PCs.tsv'
eas_genetic_PCs_path <- 'gs://path/to/EAS_genetic_PCs.tsv'
eur_genetic_PCs_path <- 'gs://path/to/EUR_genetic_PCs.tsv'
sas_genetic_PCs_path <- 'gs://path/to/SAS_genetic_PCs.tsv'
comb_genetic_PCs_path <- 'gs://path/to/COMB_genetic_PCs.tsv'
amr_genetic_PCs_path <- 'gs://path/to/AMR_genetic_PCs.tsv'

afr_phenotype_PCs_path <- 'gs://path/to/afr_pQTL_phenotype_PCs.tsv'
amr_phenotype_PCs_path <- 'gs://path/to/amr_pQTL_phenotype_PCs.tsv'
comb_phenotype_PCs_path <- 'gs://path/to/comb_pQTL_phenotype_PCs.tsv'
eas_phenotype_PCs_path <- 'gs://path/to/eas_pQTL_phenotype_PCs.tsv'
eur_phenotype_PCs_path <- 'gs://path/to/eur_pQTL_phenotype_PCs.tsv'
mid_phenotype_PCs_path <- 'gs://path/to/mid_pQTL_phenotype_PCs.tsv'
sas_phenotype_PCs_path <- 'gs://path/to/sas_pQTL_phenotype_PCs.tsv'

# copy over phenotype PCs to VM
system(paste0('gsutil cp ',mid_phenotype_PCs_path,' .'))
system(paste0('gsutil cp ',eur_phenotype_PCs_path,' .'))
system(paste0('gsutil cp ',sas_phenotype_PCs_path,' .'))
system(paste0('gsutil cp ',afr_phenotype_PCs_path,' .'))
system(paste0('gsutil cp ',amr_phenotype_PCs_path,' .'))
system(paste0('gsutil cp ',eas_phenotype_PCs_path,' .'))
system(paste0('gsutil cp ',comb_phenotype_PCs_path,' .'))


# copy over genetic PCs to workspace
system(paste0('gsutil cp ',mid_genetic_PCs_path,' .'))
system(paste0('gsutil cp ',afr_genetic_PCs_path,' .'))
system(paste0('gsutil cp ',eas_genetic_PCs_path,' .'))
system(paste0('gsutil cp ',sas_genetic_PCs_path,' .'))
system(paste0('gsutil cp ',eur_genetic_PCs_path,' .'))
system(paste0('gsutil cp ',comb_genetic_PCs_path,' .'))
system(paste0('gsutil cp ',amr_genetic_PCs_path,' .'))


comb_genetic_PCs <- fread(basename(comb_genetic_PCs_path))
mid_genetic_PCs <- fread(basename(mid_genetic_PCs_path))
afr_genetic_PCs <- fread(basename(afr_genetic_PCs_path))
eas_genetic_PCs <- fread(basename(eas_genetic_PCs_path))
sas_genetic_PCs <- fread(basename(sas_genetic_PCs_path))
eur_genetic_PCs <- fread(basename(eur_genetic_PCs_path))
amr_genetic_PCs <- fread(basename(amr_genetic_PCs_path))

comb_phenotype_PCs <- fread(basename(comb_phenotype_PCs_path))
mid_phenotype_PCs <- fread(basename(mid_phenotype_PCs_path))
eur_phenotype_PCs <- fread(basename(eur_phenotype_PCs_path))
sas_phenotype_PCs <- fread(basename(sas_phenotype_PCs_path))
afr_phenotype_PCs <- fread(basename(afr_phenotype_PCs_path))
eas_phenotype_PCs <- fread(basename(eas_phenotype_PCs_path))
amr_phenotype_PCs <- fread(basename(amr_phenotype_PCs_path))



In [ ]:
comb_merged_covariates <- comb_genetic_PCs %>% 
    inner_join(comb_phenotype_PCs,by = c('sample_id' = 'research_id')) %>% 
    dplyr::select(sample_id,everything()) %>% 
    distinct() 
mid_merged_covariates <- mid_genetic_PCs %>% 
    inner_join(mid_phenotype_PCs,by = c('sample_id' = 'research_id')) %>% 
    dplyr::select(sample_id,everything()) %>% 
    distinct() 
sas_merged_covariates <- sas_genetic_PCs %>% 
    inner_join(sas_phenotype_PCs,by = c('sample_id' = 'research_id')) %>% 
    dplyr::select(sample_id,everything()) %>% 
    distinct() 
afr_merged_covariates <- afr_genetic_PCs %>% 
    inner_join(afr_phenotype_PCs,by = c('sample_id' = 'research_id')) %>% 
    dplyr::select(sample_id,everything()) %>% 
    distinct() 
eur_merged_covariates <- eur_genetic_PCs %>% 
      inner_join(eur_phenotype_PCs,by = c('sample_id' = 'research_id')) %>%
      dplyr::select(sample_id,everything()) %>% 
      distinct()
eas_merged_covariates <- eas_genetic_PCs %>% 
      inner_join(eas_phenotype_PCs,by = c('sample_id' = 'research_id')) %>% 
      dplyr::select(sample_id,everything()) %>% 
      distinct() 
amr_merged_covariates <- amr_genetic_PCs %>% 
      inner_join(amr_phenotype_PCs,by = c('sample_id' = 'research_id')) %>% 
      dplyr::select(sample_id,everything()) %>% 
      distinct() 
comb_merged_covariates %>% filter(if_any(everything(),~is.na(.))) %>% nrow
mid_merged_covariates %>% filter(if_any(everything(),~is.na(.))) %>% nrow
afr_merged_covariates %>% filter(if_any(everything(),~is.na(.))) %>% nrow
amr_merged_covariates %>% filter(if_any(everything(),~is.na(.))) %>% nrow
eur_merged_covariates %>% filter(if_any(everything(),~is.na(.))) %>% nrow
eas_merged_covariates %>% filter(if_any(everything(),~is.na(.))) %>% nrow
sas_merged_covariates %>% filter(if_any(everything(),~is.na(.))) %>% nrow


In [ ]:
#install.packages("janitor")
library(janitor)

In [ ]:
comb_merged_covariates %>% arrange(sample_id) %>% t() %>% data.frame() %>% janitor::row_to_names(row_number = 1) %>% rownames_to_column('ID')  %>% write_tsv('AoU_COMB_pQTL_covars.tsv')
eas_merged_covariates %>% arrange(sample_id) %>% t() %>% data.frame() %>% janitor::row_to_names(row_number = 1) %>% rownames_to_column('ID')  %>% write_tsv('AoU_EAS_pQTL_covars.tsv')
afr_merged_covariates %>% arrange(sample_id) %>% t() %>% data.frame() %>% janitor::row_to_names(row_number = 1) %>% rownames_to_column('ID')  %>% write_tsv('AoU_AFR_pQTL_covars.tsv')
sas_merged_covariates %>% arrange(sample_id) %>% t() %>% data.frame() %>% janitor::row_to_names(row_number = 1) %>% rownames_to_column('ID')  %>% write_tsv('AoU_SAS_pQTL_covars.tsv')
eur_merged_covariates %>% arrange(sample_id) %>% t() %>% data.frame() %>% janitor::row_to_names(row_number = 1) %>% rownames_to_column('ID')  %>% write_tsv('AoU_EUR_pQTL_covars.tsv')
mid_merged_covariates %>% arrange(sample_id) %>% t() %>% data.frame() %>% janitor::row_to_names(row_number = 1) %>% rownames_to_column('ID')  %>% write_tsv('AoU_MID_pQTL_covars.tsv')
amr_merged_covariates %>% arrange(sample_id) %>% t() %>% data.frame() %>% janitor::row_to_names(row_number = 1) %>% rownames_to_column('ID')  %>% write_tsv('AoU_AMR_pQTL_covars.tsv')


system(paste0('gsutil cp AoU_AMR_pQTL_covars.tsv gs://path/to/pQTL/covariates/'))
system(paste0('gsutil cp AoU_COMB_pQTL_covars.tsv gs://path/to/pQTL/covariates/'))
system(paste0('gsutil cp AoU_AFR_pQTL_covars.tsv gs://path/to/pQTL/covariates/'))
system(paste0('gsutil cp AoU_MID_pQTL_covars.tsv gs://path/to/pQTL/covariates/'))
system(paste0('gsutil cp AoU_SAS_pQTL_covars.tsv gs://path/to/pQTL/covariates/'))
system(paste0('gsutil cp AoU_EAS_pQTL_covars.tsv gs://path/to/pQTL/covariates/'))
system(paste0('gsutil cp AoU_EUR_pQTL_covars.tsv gs://path/to/pQTL/covariates/'))